# Bauten ab 1970 anzeigen

Dieses Notebook lädt die Excel-Datei `data/Bauten_ab_1970.xlsx` als DataFrame und zeigt den Inhalt an.

## 1) Excel-Datei laden


In [30]:
from pathlib import Path
import pandas as pd

excel_path = Path.cwd().parent / 'data' / 'Bauten_ab_1970.xlsx'
print('Datei:', excel_path)
print('Grösse (Bytes):', excel_path.stat().st_size)

if excel_path.stat().st_size == 0:
    raise ValueError('Die Excel-Datei ist leer (0 Bytes). Bitte die Datei erneut bereitstellen.')

df = pd.read_excel(excel_path)
df

Datei: /Users/padrian/Documents/08_Tools/51_Inventar_Luzern/data/Bauten_ab_1970.xlsx
Grösse (Bytes): 21312


,Adresse,Objektbezeichnung,BJ,Architekt,BILU,Bild,Gemeinde,Typ
0,Alpenquai 12-14,NaN,1980/81,Walter Rüssli/Hans Eggstein,NaN,NaN,Luzern,Gebäude
1,Alpenquai 20-22,NaN,NaN,NaN,NaN,NaN,Luzern,Gebäude
2,Alpenquai 28-30,Verwaltungsgebäude Buchecker,1992,Walter Rüssli,NaN,NaN,Luzern,Gebäude
3,Alpenquai 34 / Landenbergstrasse 14/16,NaN,1984,NaN,NaN,NaN,Luzern,Gebäude
4,Alpenquai 36-40 / Landenbergstrasse 8-12,NaN,1985,NaN,NaN,NaN,Luzern,Gebäude
5,Alpenquai 42,NaN,1970,NaN,NaN,NaN,Luzern,Gebäude
6,Alpenstrasse 12,NaN,1980,NaN,NaN,NaN,Luzern,Gebäude
7,Bahnhofplatz 1,Aufnahmegebäude,1983-1990,Ammann/Baumann,NaN,NaN,Luzern,Gebäude
8,Baselstrasse 4,Parkhaus,NaN,NaN,NaN,NaN,Luzern,Gebäude
9,Baselstrasse 31-33,Wohnhaus,1979,NaN,NaN,NaN,Luzern,Gebäude


## 2) Gemeindeangaben bereinigen


In [31]:
mask = df['Gemeinde'].eq('Agglomeration') & df['Adresse'].notna()

def extract_gemeinde(adresse: object) -> str:
    text = str(adresse).strip()
    if ',' in text:
        return text.split(',', 1)[0].strip()
    return text.split()[0] if text else text

df.loc[mask, 'Gemeinde'] = df.loc[mask, 'Adresse'].apply(extract_gemeinde)

print('Bereinigte Einträge:', int(mask.sum()))
display(df.loc[mask, ['Gemeinde', 'Adresse']].head(20))


Bereinigte Einträge: 31


,Gemeinde,Adresse
140,Horw,"Horw, Technikum"
141,Horw,"Horw, Siedlung Stirnrüti"
142,Horw,"Horw, Gemeindehausplatz"
143,Emmen,Emmen Siedlung Benziwil
144,Emmen,Emmen Gerliswilstrasse 6
145,Emmen,Emmen Hochdorferstrasse 1
146,Emmen,Emmen Rüeggisingerstrasse 27
147,Emmen,Emmen Gerliswilstrasse 42/44
148,Emmen,Emmen Gerliswilstrasse 59
149,Emmen,"Emmen, Oberriffig 7"


## 3) Adresse bereinigen


In [32]:
import re
from typing import Optional, Tuple

addr_mask = df['Adresse'].notna()

def parse_adresse(adresse: object, gemeinde: object) -> Tuple[Optional[str], Optional[str]]:
    text = str(adresse).strip()
    if not text or text.lower() == 'nan':
        return None, None

    first = text.split('/', 1)[0].strip()
    first = first.split('\t', 1)[0].strip()

    if isinstance(gemeinde, str) and gemeinde.strip():
        prefix = gemeinde.strip() + ' '
        if first.lower().startswith(prefix.lower()):
            first = first[len(prefix):].strip()

    primary = first.split(',', 1)[0].strip()
    match = re.match(r'^(.*?)(?:\s+)(\d[\dA-Za-z\-\/\.]*)$', primary)
    if match:
        raw_number = match.group(2).strip().split(',', 1)[0].strip()
        house_number = re.match(r'^(\d+[A-Za-z]?)(?:\s*[-/].*)?$', raw_number)
        return match.group(1).strip(), house_number.group(1) if house_number else raw_number

    return primary, None

parsed = df.loc[addr_mask, ['Adresse', 'Gemeinde']].apply(lambda row: parse_adresse(row['Adresse'], row['Gemeinde']), axis=1)
df.loc[addr_mask, 'STRNAMK1_HPT'] = parsed.map(lambda x: x[0])
df.loc[addr_mask, 'DEINR'] = parsed.map(lambda x: x[1])

print('Bereinigte Adressen:', int(addr_mask.sum()))
display(df.loc[addr_mask, ['Adresse', 'STRNAMK1_HPT', 'DEINR']].head(20))


Bereinigte Adressen: 171


,Adresse,STRNAMK1_HPT,DEINR
0,Alpenquai 12-14,Alpenquai,12
1,Alpenquai 20-22,Alpenquai,20
2,Alpenquai 28-30,Alpenquai,28
3,Alpenquai 34 / Landenbergstrasse 14/16,Alpenquai,34
4,Alpenquai 36-40 / Landenbergstrasse 8-12,Alpenquai,36
5,Alpenquai 42,Alpenquai,42
6,Alpenstrasse 12,Alpenstrasse,12
7,Bahnhofplatz 1,Bahnhofplatz,1
8,Baselstrasse 4,Baselstrasse,4
9,Baselstrasse 31-33,Baselstrasse,31


## 4) Inhalt anzeigen


In [33]:
print('Form:', df.shape)
display(df.head(20))
df.info()

Form: (171, 10)


,Adresse,Objektbezeichnung,BJ,Architekt,BILU,Bild,Gemeinde,Typ,STRNAMK1_HPT,DEINR
0,Alpenquai 12-14,NaN,1980/81,Walter Rüssli/Hans Eggstein,NaN,NaN,Luzern,Gebäude,Alpenquai,12
1,Alpenquai 20-22,NaN,NaN,NaN,NaN,NaN,Luzern,Gebäude,Alpenquai,20
2,Alpenquai 28-30,Verwaltungsgebäude Buchecker,1992,Walter Rüssli,NaN,NaN,Luzern,Gebäude,Alpenquai,28
3,Alpenquai 34 / Landenbergstrasse 14/16,NaN,1984,NaN,NaN,NaN,Luzern,Gebäude,Alpenquai,34
4,Alpenquai 36-40 / Landenbergstrasse 8-12,NaN,1985,NaN,NaN,NaN,Luzern,Gebäude,Alpenquai,36
5,Alpenquai 42,NaN,1970,NaN,NaN,NaN,Luzern,Gebäude,Alpenquai,42
6,Alpenstrasse 12,NaN,1980,NaN,NaN,NaN,Luzern,Gebäude,Alpenstrasse,12
7,Bahnhofplatz 1,Aufnahmegebäude,1983-1990,Ammann/Baumann,NaN,NaN,Luzern,Gebäude,Bahnhofplatz,1
8,Baselstrasse 4,Parkhaus,NaN,NaN,NaN,NaN,Luzern,Gebäude,Baselstrasse,4
9,Baselstrasse 31-33,Wohnhaus,1979,NaN,NaN,NaN,Luzern,Gebäude,Baselstrasse,31


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 171 entries, 0 to 170
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Adresse            171 non-null    object
 1   Objektbezeichnung  92 non-null     object
 2   BJ                 143 non-null    object
 3   Architekt          60 non-null     object
 4   BILU               41 non-null     object
 5   Bild               2 non-null      object
 6   Gemeinde           171 non-null    object
 7   Typ                171 non-null    object
 8   STRNAMK1_HPT       171 non-null    object
 9   DEINR              135 non-null    object
dtypes: object(10)
memory usage: 13.5+ KB
